# YOLOv5 / YOLOv7 / YOLOv8 Low-Data Weed Detection Benchmark
**Based on**: Transfer Learning for Low-Data Multi-Class Weed Detection in Cotton Fields  
**Dataset**: CottonWeedDet12 in YOLO format

This notebook evaluating YOLOv5, YOLOv7, and YOLOv8 on the same dataset across different training data fractions (10%, 25%, 50%, 100%).

### 1. Imports

In [ ]:
import os
import sys
import json
import shutil
import random
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

### 2. Main Configuration

In [ ]:
CONFIG = {
    # Change this path according to your environment
    "DATASET_ROOT": r"c:\Users\ahmad\Desktop\computer vision\cottonweed",

    # Working/output directory
    "WORK_DIR": r"c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark",

    # Dataset classes from proposal
    "CLASS_NAMES": [
        "weed1", "weed2", "weed3", "weed4", "weed5", "weed6",
        "weed7", "weed8", "weed9", "weed10", "weed11", "weed12"
    ],

    # Low-data settings proposed in the research draft
    "DATA_FRACTIONS": [0.10, 0.25, 0.50, 1.00],

    # Model settings - YOLOv5 skipped as requested
    "MODELS": ["yolov7", "yolov8"],

    # Transfer learning modes
    # True = pretrained weights, False = train from scratch
    "PRETRAINED_OPTIONS": [True],

    # Augmentation options
    # True = standard augmentation, False = minimal augmentation
    "AUGMENTATION_OPTIONS": [True],

    # Training hyperparameters
    "IMG_SIZE": 640,
    "BATCH_SIZE": 8,
    "EPOCHS": 50,
    "PATIENCE": 25,
    "SEED": 42,
    "DEVICE": 0,   # use 0 for GPU, "cpu" for CPU

    # GitHub repositories
    "YOLOV5_REPO": "https://github.com/ultralytics/yolov5.git",
    "YOLOV7_REPO": "https://github.com/WongKinYiu/yolov7.git",

    # Pretrained weights
    "YOLOV5_WEIGHTS": "yolov5s.pt",
    "YOLOV7_WEIGHTS_URL": "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt",
    "YOLOV8_WEIGHTS": "yolov8n.pt",
}

### 3. Utility Functions

In [ ]:
def run_command(command, cwd=None):
    """Run shell command and stop if it fails."""
    print("\n[COMMAND]", command)
    result = subprocess.run(command, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {command}")


def make_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def load_json(path, default=None):
    if Path(path).exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def write_data_yaml(path, train_path, val_path, test_path, class_names):
    """Create YOLO data.yaml file."""
    content = f"""train: {train_path}
val: {val_path}
"""
    if test_path is not None and Path(test_path).exists():
        content += f"test: {test_path}\n"

    content += f"nc: {len(class_names)}\n"
    content += "names:\n"
    for i, name in enumerate(class_names):
        content += f"  {i}: {name}\n"

    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def copy_file(src, dst):
    make_dir(Path(dst).parent)
    shutil.copy2(src, dst)


def get_image_files(folder):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
    files = []
    for ext in exts:
        files.extend(list(Path(folder).glob(ext)))
    return sorted(files)


def create_subset_dataset(dataset_root, output_root, fraction, seed=42):
    """
    Create low-data subset while keeping val/test unchanged.
    Only train images are reduced.
    """
    dataset_root = Path(dataset_root)
    output_root = Path(output_root)

    train_img_dir = dataset_root / "images" / "train"
    train_lbl_dir = dataset_root / "labels" / "train"
    val_img_dir = dataset_root / "images" / "val"
    val_lbl_dir = dataset_root / "labels" / "val"
    test_img_dir = dataset_root / "images" / "test"
    test_lbl_dir = dataset_root / "labels" / "test"

    subset_name = f"subset_{int(fraction * 100)}"
    subset_root = output_root / "datasets" / subset_name

    marker_file = subset_root / "SUBSET_READY.json"
    if marker_file.exists():
        print(f"[SKIP] Dataset subset already prepared: {subset_root}")
        return subset_root

    print(f"[INFO] Creating dataset subset: {subset_name}")

    # Clear incomplete subset if exists
    if subset_root.exists():
        shutil.rmtree(subset_root)

    # Create folders
    for split in ["train", "val", "test"]:
        make_dir(subset_root / "images" / split)
        make_dir(subset_root / "labels" / split)

    # Select training images
    train_images = get_image_files(train_img_dir)
    random.seed(seed)
    random.shuffle(train_images)

    selected_count = max(1, int(len(train_images) * fraction))
    selected_train_images = train_images[:selected_count]

    # Copy selected train images and labels
    for img_path in selected_train_images:
        label_path = train_lbl_dir / f"{img_path.stem}.txt"
        copy_file(img_path, subset_root / "images" / "train" / img_path.name)
        if label_path.exists():
            copy_file(label_path, subset_root / "labels" / "train" / label_path.name)

    # Copy full validation set
    if val_img_dir.exists():
        for img_path in get_image_files(val_img_dir):
            label_path = val_lbl_dir / f"{img_path.stem}.txt"
            copy_file(img_path, subset_root / "images" / "val" / img_path.name)
            if label_path.exists():
                copy_file(label_path, subset_root / "labels" / "val" / label_path.name)

    # Copy full test set if available
    if test_img_dir.exists():
        for img_path in get_image_files(test_img_dir):
            label_path = test_lbl_dir / f"{img_path.stem}.txt"
            copy_file(img_path, subset_root / "images" / "test" / img_path.name)
            if label_path.exists():
                copy_file(label_path, subset_root / "labels" / "test" / label_path.name)

    # Create data.yaml
    test_path = subset_root / "images" / "test" if test_img_dir.exists() else None
    write_data_yaml(
        path=subset_root / "data.yaml",
        train_path=subset_root / "images" / "train",
        val_path=subset_root / "images" / "val",
        test_path=test_path,
        class_names=CONFIG["CLASS_NAMES"]
    )

    save_json(marker_file, {
        "fraction": fraction,
        "train_images_selected": selected_count,
        "created_at": str(datetime.now())
    })

    print(f"[DONE] Subset created: {subset_root}")
    return subset_root

### 4. Repository Setup

In [ ]:
def setup_repositories(work_dir):
    """Clone YOLOv5 and YOLOv7 repositories and install required packages."""
    work_dir = Path(work_dir)
    repos_dir = work_dir / "repos"
    make_dir(repos_dir)

    yolov5_dir = repos_dir / "yolov5"
    yolov7_dir = repos_dir / "yolov7"

    if not yolov5_dir.exists():
        run_command(f'git clone {CONFIG["YOLOV5_REPO"]} "{yolov5_dir}"')
    else:
        print("[SKIP] YOLOv5 repo already exists")

    if not yolov7_dir.exists():
        run_command(f'git clone {CONFIG["YOLOV7_REPO"]} "{yolov7_dir}"')
    else:
        print("[SKIP] YOLOv7 repo already exists")

    # Install packages
    run_command("pip install -q ultralytics")
    run_command("pip install -q -r requirements.txt", cwd=yolov5_dir)
    run_command("pip install -q -r requirements.txt", cwd=yolov7_dir)

    # Download YOLOv7 pretrained weights if missing
    yolov7_weights = work_dir / "weights" / "yolov7.pt"
    make_dir(yolov7_weights.parent)
    if not yolov7_weights.exists():
        # Using curl -L which is standard on Windows for downloading
        run_command(f'curl -L {CONFIG["YOLOV7_WEIGHTS_URL"]} -o "{yolov7_weights}"')
    else:
        print("[SKIP] YOLOv7 pretrained weight already exists")

    return yolov5_dir, yolov7_dir, yolov7_weights

### 5. Experiment Status / Checkpoint Management

In [ ]:
def experiment_name(model_name, fraction, pretrained, augmentation):
    pre = "pretrained" if pretrained else "scratch"
    aug = "aug" if augmentation else "noaug"
    return f"{model_name}_data{int(fraction*100)}_{pre}_{aug}"


def experiment_completed(exp_dir):
    exp_dir = Path(exp_dir)
    # Only skip if we have the final results marker
    return (exp_dir / "EXPERIMENT_DONE.json").exists()


def mark_completed(exp_dir, metrics):
    save_json(Path(exp_dir) / "EXPERIMENT_DONE.json", {
        "completed_at": str(datetime.now()),
        "metrics": metrics
    })


def find_last_checkpoint(exp_dir, model_name):
    """Find model-specific last checkpoint for resume."""
    exp_dir = Path(exp_dir)
    last = exp_dir / "weights" / "last.pt"
    if last.exists():
        return last
    
    # Fallback to best.pt as requested
    best = exp_dir / "weights" / "best.pt"
    if best.exists():
        return best
        
    return None

### 6. Train YOLOv5

In [ ]:
def train_yolov5(yolov5_dir, subset_root, exp_dir, pretrained=True, augmentation=True):
    data_yaml = Path(subset_root) / "data.yaml"
    exp_dir = Path(exp_dir)
    make_dir(exp_dir)

    last_ckpt = find_last_checkpoint(exp_dir, "yolov5")

    if last_ckpt:
        print(f"[RESUME] YOLOv5 from {last_ckpt}")
        command = f'python train.py --resume "{last_ckpt}"'
    else:
        weights = CONFIG["YOLOV5_WEIGHTS"] if pretrained else ""
        command = (
            f"python train.py "
            f"--img {CONFIG['IMG_SIZE']} "
            f"--batch {CONFIG['BATCH_SIZE']} "
            f"--epochs {CONFIG['EPOCHS']} "
            f'--data "{data_yaml}" '
            f'--weights "{weights}" '
            f'--project "{exp_dir.parent}" '
            f'--name "{exp_dir.name}" '
            f"--exist-ok "
            f"--device {CONFIG['DEVICE']} "
            f"--workers 0 "
            f"--patience {CONFIG['PATIENCE']} "
            f"--seed {CONFIG['SEED']}"
        )
        if not augmentation:
            command += " --augment False"

    run_command(command, cwd=yolov5_dir)

    best_ckpt = exp_dir / "weights" / "best.pt"
    metrics = validate_yolov5(yolov5_dir, data_yaml, best_ckpt, exp_dir)
    return metrics


def validate_yolov5(yolov5_dir, data_yaml, weights_path, exp_dir):
    val_dir = Path(exp_dir) / "validation"
    make_dir(val_dir)

    command = (
        f"python val.py "
        f'--weights "{weights_path}" '
        f'--data "{data_yaml}" '
        f"--img {CONFIG['IMG_SIZE']} "
        f"--batch {CONFIG['BATCH_SIZE']} "
        f"--task val "
        f'--project "{val_dir.parent}" '
        f"--name validation "
        f"--exist-ok "
        f"--save-json"
    )
    run_command(command, cwd=yolov5_dir)

    # YOLOv5 saves main metrics in results.csv during training.
    results_csv = Path(exp_dir) / "results.csv"
    return parse_yolo_results_csv(results_csv)

### 7. Train YOLOv7

In [ ]:
def train_yolov7(yolov7_dir, yolov7_weights, subset_root, exp_dir, pretrained=True, augmentation=True):
    data_yaml = Path(subset_root) / "data.yaml"
    exp_dir = Path(exp_dir)
    make_dir(exp_dir)

    last_ckpt = find_last_checkpoint(exp_dir, "yolov7")

    if last_ckpt:
        print(f"[RESUME] YOLOv7 from {last_ckpt}")
        command = f'python train.py --resume "{last_ckpt}"'
    else:
        weights = yolov7_weights if pretrained else ""
        hyp_file = "data/hyp.scratch.p5.yaml"

        command = (
            f"python train.py "
            f"--workers 0 "
            f"--device {CONFIG['DEVICE']} "
            f"--batch-size {CONFIG['BATCH_SIZE']} "
            f'--data "{data_yaml}" '
            f"--img {CONFIG['IMG_SIZE']} {CONFIG['IMG_SIZE']} "
            f"--cfg cfg/training/yolov7.yaml "
            f'--weights "{weights}" '
            f"--name {exp_dir.name} "
            f'--project "{exp_dir.parent}" '
            f"--hyp {hyp_file} "
            f"--epochs {CONFIG['EPOCHS']}"
        )

    # Check if best.pt already exists to avoid re-training
    best_ckpt = exp_dir / "weights" / "best.pt"
    if not best_ckpt.exists():
        run_command(command, cwd=yolov7_dir)
    else:
        print(f"\n[INFO] best.pt found in {exp_dir.name}. Skipping training command and processing results...")

    metrics = validate_yolov7(yolov7_dir, data_yaml, best_ckpt, exp_dir)
    return metrics


def validate_yolov7(yolov7_dir, data_yaml, weights_path, exp_dir):
    val_dir = Path(exp_dir) / "validation"
    make_dir(val_dir)

    command = (
        f"python test.py "
        f'--weights "{weights_path}" '
        f'--data "{data_yaml}" '
        f"--img-size {CONFIG['IMG_SIZE']} "
        f"--batch-size {CONFIG['BATCH_SIZE']} "
        f"--task val "
        f'--project "{val_dir.parent}" '
        f"--name validation "
        f"--exist-ok"
    )
    run_command(command, cwd=yolov7_dir)

    results_txt = Path(exp_dir) / "results.txt"
    return parse_yolov7_results(results_txt)

### 8. Train YOLOv8

In [ ]:
def train_yolov8(subset_root, exp_dir, pretrained=True, augmentation=True):
    from ultralytics import YOLO

    data_yaml = Path(subset_root) / "data.yaml"
    exp_dir = Path(exp_dir)
    make_dir(exp_dir)

    last_ckpt = find_last_checkpoint(exp_dir, "yolov8")

    if last_ckpt:
        print(f"[RESUME] YOLOv8 from {last_ckpt}")
        model = YOLO(str(last_ckpt))
        model.train(resume=True)
    else:
        weights = CONFIG["YOLOV8_WEIGHTS"] if pretrained else "yolov8n.yaml"
        model = YOLO(weights)

        model.train(
            data=str(data_yaml),
            imgsz=CONFIG["IMG_SIZE"],
            epochs=CONFIG["EPOCHS"],
            batch=CONFIG["BATCH_SIZE"],
            patience=CONFIG["PATIENCE"],
            device=CONFIG["DEVICE"],
            project=str(exp_dir.parent),
            name=exp_dir.name,
            exist_ok=True,
            seed=CONFIG["SEED"],
            pretrained=pretrained,
            augment=augmentation,
            plots=True,
            save=True,
            verbose=True,
            workers=0
        )

    best_ckpt = exp_dir / "weights" / "best.pt"
    model = YOLO(str(best_ckpt))
    results = model.val(data=str(data_yaml), imgsz=CONFIG["IMG_SIZE"], batch=CONFIG["BATCH_SIZE"], plots=True)

    metrics = {
        "precision": float(results.box.mp),
        "recall": float(results.box.mr),
        "map50": float(results.box.map50),
        "map50_95": float(results.box.map),
    }
    return metrics

### 9. Result Parsers

In [ ]:
def parse_yolo_results_csv(results_csv):
    """Parse YOLOv5 or YOLOv8 style results.csv when available."""
    results_csv = Path(results_csv)
    if not results_csv.exists():
        print(f"[WARN] Missing results.csv: {results_csv}")
        return {
            "precision": None,
            "recall": None,
            "map50": None,
            "map50_95": None,
        }

    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]

    possible_cols = {
        "precision": ["metrics/precision", "metrics/precision(B)", "P", "precision"],
        "recall": ["metrics/recall", "metrics/recall(B)", "R", "recall"],
        "map50": ["metrics/mAP_0.5", "metrics/mAP50(B)", "mAP@0.5", "map50"],
        "map50_95": ["metrics/mAP_0.5:0.95", "metrics/mAP50-95(B)", "mAP@0.5:0.95", "map50_95"],
    }

    metrics = {}
    for key, cols in possible_cols.items():
        value = None
        for col in cols:
            if col in df.columns:
                value = float(last[col])
                break
        metrics[key] = value

    return metrics


def parse_yolov7_results(results_txt):
    """
    YOLOv7 results parsing tries to read results.txt if present.
    """
    results_txt = Path(results_txt)
    if not results_txt.exists():
        print(f"[WARN] Missing YOLOv7 results.txt: {results_txt}")
        return {
            "precision": None,
            "recall": None,
            "map50": None,
            "map50_95": None,
        }

    try:
        lines = results_txt.read_text().strip().splitlines()
        last_line = lines[-1].split()
        nums = []
        for x in last_line:
            try:
                nums.append(float(x))
            except Exception:
                pass

        if len(nums) >= 4:
            return {
                "precision": nums[-4],
                "recall": nums[-3],
                "map50": nums[-2],
                "map50_95": nums[-1],
            }
    except Exception as e:
        print("[WARN] Could not parse YOLOv7 results:", e)

    return {
        "precision": None,
        "recall": None,
        "map50": None,
        "map50_95": None,
    }

### 10. Save and Plot Comparison Results

In [ ]:
def append_result(master_csv, row):
    master_csv = Path(master_csv)
    make_dir(master_csv.parent)

    if master_csv.exists():
        df = pd.read_csv(master_csv)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df = df.drop_duplicates(
            subset=["model", "data_fraction", "pretrained", "augmentation"],
            keep="last"
        )
    else:
        df = pd.DataFrame([row])

    df.to_csv(master_csv, index=False)
    return df


def create_comparison_charts(master_csv, output_dir):
    master_csv = Path(master_csv)
    output_dir = Path(output_dir)
    make_dir(output_dir)

    if not master_csv.exists():
        print("[WARN] No master results found for charts.")
        return

    df = pd.read_csv(master_csv)
    df = df.sort_values(["data_fraction", "model"])

    metric_map = {
        "precision": "Precision",
        "recall": "Recall",
        "map50": "mAP@0.5",
        "map50_95": "mAP@0.5:0.95",
    }

    for metric, title in metric_map.items():
        if metric not in df.columns:
            continue

        plt.figure(figsize=(10, 6))
        for model_name in sorted(df["model"].dropna().unique()):
            sub = df[df["model"] == model_name]
            plt.plot(sub["data_fraction"] * 100, sub[metric], marker="o", label=model_name)

        plt.xlabel("Training Data Used (%)")
        plt.ylabel(title)
        plt.title(f"{title} Comparison Across YOLO Models")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / f"comparison_{metric}.png", dpi=300)
        plt.close()

    # Grouped bar chart for full-data performance
    full_df = df[df["data_fraction"] == 1.0]
    if len(full_df) > 0:
        metrics = ["precision", "recall", "map50", "map50_95"]
        full_plot = full_df.set_index("model")[metrics]
        ax = full_plot.plot(kind="bar", figsize=(11, 6))
        ax.set_ylabel("Score")
        ax.set_title("Full Training Data Performance Comparison")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(output_dir / "full_data_grouped_bar_comparison.png", dpi=300)
        plt.close()

    print(f"[DONE] Charts saved in: {output_dir}")

### 11. Main Benchmark Runner

In [ ]:
def run_benchmark():
    work_dir = Path(CONFIG["WORK_DIR"])
    make_dir(work_dir)

    dataset_root = Path(CONFIG["DATASET_ROOT"])
    if not dataset_root.exists():
        raise FileNotFoundError(
            f"Dataset root not found: {dataset_root}\n"
            "Please update CONFIG['DATASET_ROOT'] according to your dataset location."
        )

    # Save experiment config
    save_json(work_dir / "benchmark_config.json", CONFIG)

    # Setup repositories
    yolov5_dir, yolov7_dir, yolov7_weights = setup_repositories(work_dir)

    master_csv = work_dir / "comparison_results" / "all_model_results.csv"
    charts_dir = work_dir / "comparison_results" / "charts"

    for fraction in CONFIG["DATA_FRACTIONS"]:
        subset_root = create_subset_dataset(
            dataset_root=dataset_root,
            output_root=work_dir,
            fraction=fraction,
            seed=CONFIG["SEED"]
        )

        for pretrained in CONFIG["PRETRAINED_OPTIONS"]:
            for augmentation in CONFIG["AUGMENTATION_OPTIONS"]:
                for model_name in CONFIG["MODELS"]:
                    exp_name = experiment_name(model_name, fraction, pretrained, augmentation)
                    exp_dir = work_dir / "runs" / exp_name

                    print("\n" + "=" * 80)
                    print(f"Starting Experiment: {exp_name}")
                    print("=" * 80)

                    if experiment_completed(exp_dir):
                        print(f"[SKIP] Completed already: {exp_name}")
                        done_data = load_json(exp_dir / "EXPERIMENT_DONE.json", default={})
                        metrics = done_data.get("metrics", {})
                    else:
                        if model_name == "yolov5":
                            metrics = train_yolov5(
                                yolov5_dir=yolov5_dir,
                                subset_root=subset_root,
                                exp_dir=exp_dir,
                                pretrained=pretrained,
                                augmentation=augmentation
                            )
                        elif model_name == "yolov7":
                            metrics = train_yolov7(
                                yolov7_dir=yolov7_dir,
                                yolov7_weights=yolov7_weights,
                                subset_root=subset_root,
                                exp_dir=exp_dir,
                                pretrained=pretrained,
                                augmentation=augmentation
                            )
                        elif model_name == "yolov8":
                            metrics = train_yolov8(
                                subset_root=subset_root,
                                exp_dir=exp_dir,
                                pretrained=pretrained,
                                augmentation=augmentation
                            )
                        else:
                            raise ValueError(f"Unsupported model: {model_name}")

                        mark_completed(exp_dir, metrics)

                    row = {
                        "model": model_name,
                        "data_fraction": fraction,
                        "pretrained": pretrained,
                        "augmentation": augmentation,
                        "precision": metrics.get("precision"),
                        "recall": metrics.get("recall"),
                        "map50": metrics.get("map50"),
                        "map50_95": metrics.get("map50_95"),
                        "experiment_dir": str(exp_dir),
                        "timestamp": str(datetime.now())
                    }

                    df = append_result(master_csv, row)
                    print("\n[CURRENT RESULT]")
                    print(pd.DataFrame([row]))

                    create_comparison_charts(master_csv, charts_dir)

    print("\n" + "=" * 80)
    print("BENCHMARK COMPLETED")
    print(f"Results CSV: {master_csv}")
    print(f"Charts: {charts_dir}")
    print("=" * 80)

if __name__ == "__main__":
    run_benchmark()